# GNN Timing Predictor - Training on Google Colab

Train your timing violation predictor with GPU acceleration.

**Expected time:** 20-30 minutes

**Expected performance:** 0.96+ ROC-AUC

## Step 1: Upload Dataset Package

Upload `gnn_timing_pack.zip` (created with `python scripts/prepare_colab_pack.py`)

In [ ]:
from google.colab import files
import os

print("📤 Upload gnn_timing_pack.zip...")
uploaded = files.upload()
print("✅ Upload complete!")

## Step 2: Extract Package

In [ ]:
!unzip -o -q gnn_timing_pack.zip
!ls -la
print("✅ Package extracted!")

## Step 3: Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q torch-geometric
!pip install -q loguru pyyaml scikit-learn matplotlib tensorboard
print("✅ Dependencies installed!")

## Step 4: Train Model

Training takes ~20-30 minutes on GPU.

In [ ]:
!python -m src.training.train \
    --config experiments/configs/default.yaml \
    --data_dir data/processed/timing_predict \
    --checkpoint_dir experiments/checkpoints \
    --log_dir experiments/logs \
    --gpu

## Step 5: Monitor Training (Optional)

Run in separate cell while training:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir experiments/logs

## Step 6: Compute Optimal Threshold (Baseline Comparison)

**Note:** RankSTA doesn't use thresholds. This computes the optimal threshold for baseline comparison only.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import f1_score
import sys

sys.path.append('.')
from src.models.timing_gnn import HeterogeneousTimingGNN
from src.data.dataset import TimingDataset

# Load trained model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load('experiments/checkpoints/best_model.pth', map_location=device)

model_cfg = checkpoint.get('config', {}).get('model', {})
model = HeterogeneousTimingGNN(
    in_channels=model_cfg.get('in_channels', 10),
    hidden_channels=model_cfg.get('hidden_channels', 128),
    num_classes=2,
    num_layers=model_cfg.get('num_layers', 3),
    heads=model_cfg.get('attention_heads', 4),
    dropout=model_cfg.get('dropout', 0.2),
    edge_dim=model_cfg.get('edge_dim', 3)
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Load validation set
val_dataset = TimingDataset(root='data/processed/timing_predict', split='val')

# Get predictions
all_probs, all_labels = [], []
with torch.no_grad():
    for data in val_dataset:
        data = data.to(device)
        logits = model(data)
        probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        labels = data.y.cpu().numpy()
        
        mask = labels >= 0
        all_probs.extend(probs[mask])
        all_labels.extend(labels[mask])

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Grid search for optimal threshold
best_f1 = 0
best_threshold = 0.5

for thresh in np.arange(0.1, 1.0, 0.1):
    preds = (all_probs >= thresh).astype(int)
    f1 = f1_score(all_labels, preds, zero_division=0)
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thresh

print(f"\n" + "="*50)
print(f" OPTIMAL THRESHOLD: {best_threshold:.1f}")
print(f" F1-SCORE: {best_f1:.4f}")
print("="*50)
print(f"\nUpdate paper with: θ* = {best_threshold:.2f}")

## Step 7: Check Final Metrics

In [ ]:
import torch

checkpoint = torch.load('experiments/checkpoints/best_model.pth', map_location='cpu')
metrics = checkpoint.get('metrics', {})

print("\n" + "="*50)
print("FINAL MODEL METRICS")
print("="*50)
for key, value in metrics.items():
    print(f"{key}: {value}")
print("="*50)

## Step 8: Download Trained Model

In [ ]:
from google.colab import files
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_zip = f"gnn_timing_results_{timestamp}.zip"

!zip -r {output_zip} experiments/checkpoints/best_model.pth experiments/logs/

files.download(output_zip)
print(f"✅ Downloaded {output_zip}")
print("\nNext steps:")
print("1. Extract best_model.pth to experiments/checkpoints/")
print("2. Run: python scripts/paper/FINAL_generate_tables.py")
print("3. All tables auto-updated!")